In [ ]:
# Cell 1 — Setup + data load (S&P 100, 2015->now)
import pandas as pd, yfinance as yf
from pathlib import Path
from backtest.data import DataPanel

DATA_DIR = Path(r"C:\Users\User\backtest_engine\backtest_engine2\data")
PX_PATH, VOL_PATH = DATA_DIR / "sp100_prices.parquet", DATA_DIR / "sp100_volume.parquet"
START = "2015-01-01"

# Current S&P 100 (OEX) members. Survivorship-biased by construction (logged in registry).
SP100 = ["AAPL","ABBV","ABT","ACN","ADBE","AIG","AMD","AMGN","AMT","AMZN","AVGO","AXP",
 "BA","BAC","BK","BKNG","BLK","BMY","BRK-B","C","CAT","CHTR","CL","CMCSA","COF","COP",
 "COST","CRM","CSCO","CVS","CVX","DHR","DIS","DOW","DUK","EMR","FDX","GD","GE","GILD",
 "GM","GOOG","GOOGL","GS","HD","HON","IBM","INTC","INTU","JNJ","JPM","KO","LIN","LLY",
 "LMT","LOW","MA","MCD","MDLZ","MDT","MET","META","MMM","MO","MRK","MS","MSFT","NEE",
 "NFLX","NKE","NVDA","ORCL","PEP","PFE","PG","PM","PYPL","QCOM","RTX","SBUX","SCHW",
 "SO","SPG","T","TGT","TMO","TMUS","TSLA","TXN","UNH","UNP","UPS","USB","V","VZ","WFC",
 "WMT","XOM"]

if PX_PATH.exists() and VOL_PATH.exists():
    prices = pd.read_parquet(PX_PATH); volume = pd.read_parquet(VOL_PATH)
    print(f"Loaded cached parquet: {prices.shape[1]} tickers.")
else:
    raw = yf.download(SP100, start=START, auto_adjust=True, progress=False, threads=True)
    prices = raw["Close"].copy(); volume = raw["Volume"].copy()
    # identical index & columns required by DataPanel; drop tickers that returned nothing
    good = [c for c in prices.columns if prices[c].notna().any()]
    prices = prices[good].sort_index(); volume = volume.reindex(index=prices.index, columns=good)
    prices.index.name = volume.index.name = "Date"
    DATA_DIR.mkdir(exist_ok=True)
    prices.to_parquet(PX_PATH); volume.to_parquet(VOL_PATH)
    print(f"Downloaded {len(good)}/{len(SP100)} tickers -> cached parquet.")

panel = DataPanel(prices, volume=volume, check_outliers=True)
print(f"Bars:   {len(panel.dates)}")
print(f"Assets: {len(panel.assets_all)}")
print(f"Range:  {panel.dates[0].date()} -> {panel.dates[-1].date()}")
print(f"NaN fraction in prices: {prices.isna().mean().mean():.3%}")
prices.tail(3).iloc[:, :6]


In [ ]:
# Cell 2 — Strategy class (short-term cross-sectional reversal)
import pandas as pd
from backtest.strategy import Strategy

class ShortTermReversal(Strategy):
    """Daily dollar-neutral short-term reversal: short recent winners, long recent
    losers on trailing k-day return, demeaned across the cross-section."""
    rebalance_frequency = "daily"

    def __init__(self, lookback: int = 5):
        self.lookback = lookback

    def __repr__(self):
        return f"ShortTermReversal(lookback={self.lookback})"

    def required_data(self):
        return {"prices": None}

    def generate_weights(self, data, t):
        k = self.lookback
        px = data.prices                       # loc[:t] inclusive — no lookahead
        if len(px) < k + 1:
            return pd.Series(0.0, index=data.assets)
        cols = data.assets                      # eligible names at t
        ret = px[cols].iloc[-1] / px[cols].iloc[-1 - k] - 1.0
        ret = ret.dropna()
        if len(ret) < 2:
            return pd.Series(0.0, index=data.assets)
        sig = -ret                              # reversal
        sig = sig - sig.mean()                  # cross-sectional demean -> dollar-neutral
        gross = sig.abs().sum()
        if gross == 0:
            return pd.Series(0.0, index=data.assets)
        return sig / gross                      # gross exposure = 1.0

strat = ShortTermReversal(lookback=5)
print(strat)
rb = strat.rebalance_dates(panel.dates)
print(f"Rebalances: {len(rb)}  (~{len(rb) / (len(panel.dates) / 252):.0f}/yr)")


In [ ]:
# Cell 3 — Costs + liquidity (realistic S&P 100 large-cap stack)
from backtest.costs import (
    Commission, Spread, MarketImpact, ShortBorrow,
    CompositeCostModel, LiquidityCap,
)

costs = CompositeCostModel([
    Commission(bps=1),                                  # 1 bp institutional commission
    Spread(half_bps=1),                                 # 1 bp half-spread (2 bp full)
    MarketImpact(k=10, kind="sqrt", adv_lookback=20),   # sqrt impact, 20-bar ADV
    ShortBorrow(annual_bps=50, trading_days=252),       # 0.5%/yr easy-to-borrow
])
liq = LiquidityCap(cap_pct=0.10, adv_lookback=20)       # 10% of ADV

print("Cost components:", [type(m).__name__ for m in costs.models])
print(f"LiquidityCap: cap_pct={liq.cap_pct}, adv_lookback={liq.adv_lookback}")
print(f"Panel has volume: {panel.has_field('volume')}")


In [ ]:
# Cell 4 — Single-path engine run (raw reversal backtest)
from backtest.engine import Engine

WARMUP_BARS = 21
engine = Engine(panel, strat, costs=costs, liquidity=liq, initial_capital=1_000_000)
result = engine.run(start=panel.dates[WARMUP_BARS])

print(f"Bars:                {len(result.equity_curve)}")
print(f"Rebalances:          {result.metadata['n_rebalances']}")
print(f"Date range:          {result.metadata['start'].date()} -> {result.metadata['end'].date()}")
print(f"Initial:             ${result.metadata['initial_capital']:>14,.0f}")
print(f"Final equity:        ${result.equity_curve.iloc[-1]:>14,.0f}")
print(f"Total return:        {(result.equity_curve.iloc[-1]/result.metadata['initial_capital']-1):+.2%}")
print(f"Total costs:         ${result.costs.sum():>14,.0f}")
print(f"Max position weight: {result.weights.abs().max().max():.2%}")
print(f"Max gross exposure:  {result.weights.abs().sum(axis=1).max():.2%}")


In [ ]:
# Cell 5 — Risk overlay: 3% per-name cap (new trial), compared vs raw
from backtest.risk import RiskConfig

class ShortTermReversalCapped(ShortTermReversal):
    risk = RiskConfig(max_position=0.03, max_gross=1.0, max_net=1.0)
    def __repr__(self):
        return f"ShortTermReversalCapped(lookback={self.lookback}, max_position=0.03)"

strat_cap = ShortTermReversalCapped(lookback=5)
result_cap = Engine(panel, strat_cap, costs=costs, liquidity=liq,
                    initial_capital=1_000_000).run(start=panel.dates[WARMUP_BARS])

def line(name, r):
    eq0 = r.metadata["initial_capital"]
    print(f"{name:<8} | maxpos {r.weights.abs().max().max():>6.2%} | "
          f"gross {r.weights.abs().sum(axis=1).max():>6.2%} | "
          f"mean|w| {r.weights.abs().replace(0,float('nan')).mean().mean():>5.2%} | "
          f"net {(r.equity_curve.iloc[-1]/eq0-1):>+7.2%} | "
          f"gross {(r.gross_pnl.sum()/eq0):>+7.2%} | "
          f"costs ${r.costs.sum():>9,.0f}")

print("trial    | max/name | gross-exp | avg pos | NET ret | GROSS ret | costs")
line("raw20%", result)
line("cap3%",  result_cap)


In [ ]:
# Cell 6 — Metrics report (Stage 7) on the 3%-capped trial
from backtest.metrics import compute_metrics, sharpe_ratio

rep = compute_metrics(
    result_cap.returns, result_cap.equity_curve,
    costs=result_cap.costs, trades=result_cap.trades,
    ann_factor=252, ci_level=0.95,
)
years = (result_cap.equity_curve.index[-1] - result_cap.equity_curve.index[0]).days / 365.25
gross_ret = result_cap.gross_pnl / result_cap.equity_curve.shift(1)
gross_sharpe = sharpe_ratio(gross_ret, ann_factor=252)

print(rep)
print()
print(f"NET  Sharpe:   {rep.sharpe:+.3f}  (95% CI [{rep.sharpe_ci_low:+.2f}, {rep.sharpe_ci_high:+.2f}])")
print(f"GROSS Sharpe:  {gross_sharpe:+.3f}   <- edge before costs")
print(f"Turnover:      {rep.turnover:.1f}x total  ->  {rep.turnover/years:.1f}x per year")
print(f"Margin:        {rep.margin*1e4:+.2f} bps net PnL per $ traded")
print(f"PSR(0):        {rep.psr:.3f}   {'sig' if rep.psr>0.95 else 'NOT sig'} at 95%")
print(f"MinTRL(0):     {rep.min_trl:.0f} bars vs sample {rep.n_obs}  "
      f"({'powered' if rep.min_trl<rep.n_obs else 'underpowered'})")


In [ ]:
# Cell 7 — Data load: S&P 400 MidCap (illiquidity-premium test universe)
import pandas as pd, requests, io, yfinance as yf
from pathlib import Path
from backtest.data import DataPanel

DATA_DIR = Path(r"C:\Users\User\backtest_engine\backtest_engine2\data")
PX_PATH, VOL_PATH = DATA_DIR / "sp400_prices.parquet", DATA_DIR / "sp400_volume.parquet"
START = "2015-01-01"

if PX_PATH.exists() and VOL_PATH.exists():
    prices_mid = pd.read_parquet(PX_PATH); volume_mid = pd.read_parquet(VOL_PATH)
    print(f"Loaded cached parquet: {prices_mid.shape[1]} tickers.")
else:
    h = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
    r = requests.get("https://en.wikipedia.org/wiki/List_of_S%26P_400_companies", headers=h, timeout=30)
    r.raise_for_status()
    syms = pd.read_html(io.StringIO(r.text))[0]["Symbol"].astype(str).str.replace(".", "-", regex=False).tolist()
    raw = yf.download(syms, start=START, auto_adjust=True, progress=False, threads=True)
    prices_mid = raw["Close"].copy(); volume_mid = raw["Volume"].copy()
    good = [c for c in prices_mid.columns if prices_mid[c].notna().any()]
    prices_mid = prices_mid[good].sort_index(); volume_mid = volume_mid.reindex(index=prices_mid.index, columns=good)
    prices_mid.index.name = volume_mid.index.name = "Date"
    DATA_DIR.mkdir(exist_ok=True)
    prices_mid.to_parquet(PX_PATH); volume_mid.to_parquet(VOL_PATH)
    print(f"Downloaded {len(good)}/{len(syms)} tickers -> cached parquet.")

panel_mid = DataPanel(prices_mid, volume=volume_mid, check_outliers=True)
print(f"Bars:   {len(panel_mid.dates)}")
print(f"Assets: {len(panel_mid.assets_all)}")
print(f"Range:  {panel_mid.dates[0].date()} -> {panel_mid.dates[-1].date()}")
print(f"NaN fraction in prices: {prices_mid.isna().mean().mean():.3%}")

# Liquidity check: median daily $-volume per name, full-sample (diagnostic only, not a filter)
dollar_vol = (prices_mid * volume_mid).median()
q = dollar_vol.quantile([0.05, 0.25, 0.5, 0.75, 0.95]) / 1e6
print("Median daily $-volume per name ($M):")
print(f"  p5={q.iloc[0]:.1f}  p25={q.iloc[1]:.1f}  median={q.iloc[2]:.1f}  p75={q.iloc[3]:.1f}  p95={q.iloc[4]:.1f}")
print(f"  names under $5M/day: {(dollar_vol < 5e6).sum()} of {len(dollar_vol)}")


In [ ]:
# Cell 8 — Ranked + clipped reversal strategy (trial #002)
import pandas as pd
from backtest.risk import RiskConfig

class ReversalRanked(ShortTermReversal):
    """Reversal on cross-sectional RANK of clipped k-day return. Robust to data
    artifacts: daily returns winsorized to +-clip before compounding, then ranked."""
    risk = RiskConfig(max_position=0.03, max_gross=1.0, max_net=1.0)

    def __init__(self, lookback: int = 5, clip: float = 0.5):
        self.lookback = lookback
        self.clip = clip

    def __repr__(self):
        return f"ReversalRanked(lookback={self.lookback}, clip={self.clip})"

    def generate_weights(self, data, t):
        k = self.lookback
        px = data.prices
        if len(px) < k + 1:
            return pd.Series(0.0, index=data.assets)
        cols = data.assets
        dr = px[cols].pct_change().iloc[-k:].clip(-self.clip, self.clip)  # clipped daily rets
        valid = dr.notna().all()                                          # full k-window only
        ret = ((1.0 + dr).prod() - 1.0)[valid]
        if len(ret) < 2:
            return pd.Series(0.0, index=data.assets)
        r = ret.rank()                          # cross-sectional rank (1..N)
        sig = -(r - r.mean())                    # reversal: losers long, winners short
        gross = sig.abs().sum()
        if gross == 0:
            return pd.Series(0.0, index=data.assets)
        return sig / gross

strat_rank = ReversalRanked(lookback=5, clip=0.5)
print(strat_rank)
# sanity: weights at the last date (pre-engine, pre-risk-cap)
v = panel_mid.as_of(panel_mid.dates[-1])
w = strat_rank.generate_weights(v, panel_mid.dates[-1])
print(f"nonzero names: {(w!=0).sum()}  gross: {w.abs().sum():.3f}  net: {w.sum():+.3e}  max|w|: {w.abs().max():.3%}")


In [ ]:
# Cell 9 — Costs: mid-cap realistic stack
from backtest.costs import (
    Commission, Spread, MarketImpact, ShortBorrow,
    CompositeCostModel, LiquidityCap,
)

costs_mid = CompositeCostModel([
    Commission(bps=1.5),                                # 1.5 bp commission
    Spread(half_bps=6),                                 # 6 bp half-spread (12 bp full) — mid-cap
    MarketImpact(k=12, kind="sqrt", adv_lookback=20),   # higher impact than large-cap
    ShortBorrow(annual_bps=150, trading_days=252),      # 1.5%/yr borrow
])
liq_mid = LiquidityCap(cap_pct=0.08, adv_lookback=20)   # 8% of ADV

print("Cost components:", [type(m).__name__ for m in costs_mid.models])
print(f"LiquidityCap: cap_pct={liq_mid.cap_pct}, adv_lookback={liq_mid.adv_lookback}")
print(f"panel_mid has volume: {panel_mid.has_field('volume')}")


In [ ]:
# Cell 10 — Engine run: ranked reversal on mid-caps (trial #002)
result_mid = Engine(panel_mid, strat_rank, costs=costs_mid, liquidity=liq_mid,
                    initial_capital=1_000_000).run(start=panel_mid.dates[WARMUP_BARS])

print("trial       | max/name | gross-exp | avg pos | NET ret | GROSS ret | costs")
line("001 cap3%", result_cap)   # S&P100 raw-magnitude, large-cap costs
line("002 rank",  result_mid)   # S&P400 ranked+clipped, mid-cap costs


In [ ]:
# Cell 11 — Metrics on trial #002 (mid-cap ranked reversal)
rep_mid = compute_metrics(
    result_mid.returns, result_mid.equity_curve,
    costs=result_mid.costs, trades=result_mid.trades,
    ann_factor=252, ci_level=0.95,
)
years_m = (result_mid.equity_curve.index[-1] - result_mid.equity_curve.index[0]).days / 365.25
gross_ret_m = result_mid.gross_pnl / result_mid.equity_curve.shift(1)
gross_sharpe_m = sharpe_ratio(gross_ret_m, ann_factor=252)

print(rep_mid)
print()
print(f"NET   Sharpe:  {rep_mid.sharpe:+.3f}  (95% CI [{rep_mid.sharpe_ci_low:+.2f}, {rep_mid.sharpe_ci_high:+.2f}])")
print(f"GROSS Sharpe:  {gross_sharpe_m:+.3f}   <- is the edge real?")
print(f"Turnover:      {rep_mid.turnover:.0f}x total -> {rep_mid.turnover/years_m:.0f}x per year")
print(f"Margin:        {rep_mid.margin*1e4:+.2f} bps net PnL per $ traded")
print(f"PSR(0):        {rep_mid.psr:.3f}   GROSS would need cost cut to realize")


In [ ]:
# Cell 12 — Decay + ranked + clipped reversal (trial #003)
import pandas as pd

class ReversalRankedDecay(ReversalRanked):
    """Ranked reversal with linear signal decay over `decay` days to cut turnover.
    Stateless: recomputes the ranked signal for each lag, weights it (decay-j)."""
    def __init__(self, lookback: int = 5, clip: float = 0.5, decay: int = 5):
        super().__init__(lookback, clip)
        self.decay = decay

    def __repr__(self):
        return f"ReversalRankedDecay(lookback={self.lookback}, clip={self.clip}, decay={self.decay})"

    def _ranked_sig(self, px, cols):
        k = self.lookback
        if len(px) < k + 1:
            return None
        dr = px[cols].pct_change().iloc[-k:].clip(-self.clip, self.clip)
        ret = ((1.0 + dr).prod() - 1.0)[dr.notna().all()]
        if len(ret) < 2:
            return None
        r = ret.rank()
        sig = -(r - r.mean())
        g = sig.abs().sum()
        return sig / g if g > 0 else None

    def generate_weights(self, data, t):
        px, cols = data.prices, data.assets
        acc = None
        for j in range(self.decay):
            sub = px.iloc[:len(px) - j] if j > 0 else px
            s = self._ranked_sig(sub, cols)
            if s is None:
                continue
            w = (self.decay - j) * s
            acc = w if acc is None else acc.add(w, fill_value=0.0)
        if acc is None:
            return pd.Series(0.0, index=data.assets)
        g = acc.abs().sum()
        return acc / g if g > 0 else pd.Series(0.0, index=data.assets)

strat_decay = ReversalRankedDecay(lookback=5, clip=0.5, decay=5)
print(strat_decay)

# turnover proxy: L1 weight change over one day, decay vs non-decay
v0, v1 = panel_mid.as_of(panel_mid.dates[-1]), panel_mid.as_of(panel_mid.dates[-2])
def l1(strat):
    a = strat.generate_weights(v0, panel_mid.dates[-1])
    b = strat.generate_weights(v1, panel_mid.dates[-2])
    return (a.subtract(b, fill_value=0.0)).abs().sum()
print(f"1-day L1 turnover  ranked(no decay): {l1(strat_rank):.3f}")
print(f"1-day L1 turnover  decay=5:          {l1(strat_decay):.3f}")


In [ ]:
# Cell 13 — Engine run: decay variant (trial #003)
result_decay = Engine(panel_mid, strat_decay, costs=costs_mid, liquidity=liq_mid,
                      initial_capital=1_000_000).run(start=panel_mid.dates[WARMUP_BARS])

print("trial        | max/name | gross-exp | avg pos | NET ret | GROSS ret | costs")
line("002 rank",  result_mid)
line("003 decay", result_decay)


In [ ]:
# Cell 14 — Metrics on trial #003 (decay)
rep_dec = compute_metrics(
    result_decay.returns, result_decay.equity_curve,
    costs=result_decay.costs, trades=result_decay.trades,
    ann_factor=252, ci_level=0.95,
)
years_d = (result_decay.equity_curve.index[-1] - result_decay.equity_curve.index[0]).days / 365.25
gross_sharpe_d = sharpe_ratio(result_decay.gross_pnl / result_decay.equity_curve.shift(1), ann_factor=252)

print(rep_dec)
print()
print(f"NET   Sharpe:  {rep_dec.sharpe:+.3f}  (95% CI [{rep_dec.sharpe_ci_low:+.2f}, {rep_dec.sharpe_ci_high:+.2f}])")
print(f"GROSS Sharpe:  {gross_sharpe_d:+.3f}")
print(f"Turnover:      {rep_dec.turnover/years_d:.0f}x per year  (was 143x)")
print(f"Margin:        {rep_dec.margin*1e4:+.2f} bps net PnL per $ traded")
print(f"PSR(0):        {rep_dec.psr:.3f}   {'SIG' if rep_dec.psr>0.95 else 'not sig'} at 95%")
print(f"MinTRL(0):     {rep_dec.min_trl:.0f} bars vs sample {rep_dec.n_obs}")


In [ ]:
# Cell 15 — GICS sector map for S&P 400 (static metadata for neutralization)
import pandas as pd, requests, io
h = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
r = requests.get("https://en.wikipedia.org/wiki/List_of_S%26P_400_companies", headers=h, timeout=30)
r.raise_for_status()
tbl = pd.read_html(io.StringIO(r.text))[0]
sec_col = [c for c in tbl.columns if "Sector" in str(c)][0]
tbl["Symbol"] = tbl["Symbol"].astype(str).str.replace(".", "-", regex=False)
sector_map = dict(zip(tbl["Symbol"], tbl[sec_col].astype(str)))

# coverage vs our actual panel tickers
covered = sum(t in sector_map for t in panel_mid.assets_all)
print(f"Sector map: {len(sector_map)} symbols | covers {covered}/{len(panel_mid.assets_all)} panel names")
print(pd.Series({t: sector_map.get(t, "UNK") for t in panel_mid.assets_all}).value_counts())


In [ ]:
# Cell 16 — Sector-neutral ranked+decay reversal (trial #004); pct_change fix + neutralize flag
import pandas as pd

class ReversalNeutral(ReversalRankedDecay):
    """Adds within-sector demeaning (industry neutralization) to the decayed ranked
    signal. neutralize=False reproduces the fixed-decay baseline for clean attribution."""
    def __init__(self, lookback=5, clip=0.5, decay=5, sectors=None, neutralize=True):
        super().__init__(lookback, clip, decay)
        self.sectors = sectors or {}
        self.neutralize = neutralize

    def __repr__(self):
        return (f"ReversalNeutral(lookback={self.lookback}, clip={self.clip}, "
                f"decay={self.decay}, neutralize={self.neutralize})")

    def _ranked_sig(self, px, cols):
        k = self.lookback
        if len(px) < k + 1:
            return None
        dr = px[cols].pct_change(fill_method=None).iloc[-k:].clip(-self.clip, self.clip)
        ret = ((1.0 + dr).prod() - 1.0)[dr.notna().all()]
        if len(ret) < 2:
            return None
        r = ret.rank()
        sig = -(r - r.mean())
        g = sig.abs().sum()
        return sig / g if g > 0 else None

    def generate_weights(self, data, t):
        px, cols = data.prices, data.assets
        acc = None
        for j in range(self.decay):
            sub = px.iloc[:len(px) - j] if j > 0 else px
            s = self._ranked_sig(sub, cols)
            if s is None:
                continue
            w = (self.decay - j) * s
            acc = w if acc is None else acc.add(w, fill_value=0.0)
        if acc is None:
            return pd.Series(0.0, index=data.assets)
        if self.neutralize:
            sec = pd.Series({n: self.sectors.get(n, "UNK") for n in acc.index})
            acc = acc - acc.groupby(sec).transform("mean")   # within-sector demean
        g = acc.abs().sum()
        return acc / g if g > 0 else pd.Series(0.0, index=data.assets)

strat_decay_fix = ReversalNeutral(sectors=sector_map, neutralize=False)  # fixed baseline
strat_neutral   = ReversalNeutral(sectors=sector_map, neutralize=True)   # +sector neutral
print(strat_neutral)

# sanity: per-sector net exposure at last date, neutral vs not
v = panel_mid.as_of(panel_mid.dates[-1]); d = panel_mid.dates[-1]
sec = pd.Series({n: sector_map.get(n, "UNK") for n in panel_mid.assets_all})
for name, st in [("baseline", strat_decay_fix), ("neutral", strat_neutral)]:
    w = st.generate_weights(v, d)
    by_sec = w.groupby(sec.reindex(w.index)).sum()
    print(f"{name:>8}: max |sector net| = {by_sec.abs().max():.4f}  gross={w.abs().sum():.3f}")


In [ ]:
# Cell 17 — Run fixed-decay baseline + sector-neutral; full comparison
result_decay_fix = Engine(panel_mid, strat_decay_fix, costs=costs_mid, liquidity=liq_mid,
                          initial_capital=1_000_000).run(start=panel_mid.dates[WARMUP_BARS])
result_neutral = Engine(panel_mid, strat_neutral, costs=costs_mid, liquidity=liq_mid,
                        initial_capital=1_000_000).run(start=panel_mid.dates[WARMUP_BARS])

def summ(name, r):
    rep = compute_metrics(r.returns, r.equity_curve, costs=r.costs, trades=r.trades, ann_factor=252)
    yrs = (r.equity_curve.index[-1] - r.equity_curve.index[0]).days / 365.25
    gS = sharpe_ratio(r.gross_pnl / r.equity_curve.shift(1), ann_factor=252)
    print(f"{name:<14} | NETret {r.equity_curve.iloc[-1]/1e6-1:>+7.2%} | "
          f"netSh {rep.sharpe:>+5.2f} | grossSh {gS:>+5.2f} | "
          f"vol {rep.ann_vol:>5.1%} | maxDD {rep.max_drawdown:>5.1%} | "
          f"turn {rep.turnover/yrs:>4.0f}x | PSR {rep.psr:.2f}")

print("trial          | net ret  | netSh | grossSh | vol   | maxDD | turn | PSR")
summ("003 decay(old)", result_decay)       # original, pad-fill bug
summ("003f decay-fix", result_decay_fix)   # fill_method=None
summ("004 neutral",    result_neutral)     # + sector neutral


In [ ]:
# Cell 18 — Crypto daily panel from Binance hourly top-50 pool
import pandas as pd
from backtest.data import DataPanel

p = r"C:\Users\User\backtest_engine\backtest_engine2\data\binance_hourly_top50_pool.parquet"
df = pd.read_parquet(p)

# drop stablecoins / metal-pegged (no reversal signal, pollute the cross-section)
EXCLUDE = {"BFUSDUSDT","RLUSDUSDT","USD1USDT","USDEUSDT","USDSUSDT","XAUTUSDT","PAXGUSDT"}
df = df[~df["symbol"].isin(EXCLUDE)].copy()
df["coin_vol"] = df["quote_volume"] / df["close"]          # quote($)->coin count for engine

close = df.pivot_table(index="open_time", columns="symbol", values="close")
cvol  = df.pivot_table(index="open_time", columns="symbol", values="coin_vol")
close.index = pd.to_datetime(close.index).tz_localize(None)
cvol.index  = pd.to_datetime(cvol.index).tz_localize(None)

prices_crypto = close.resample("1D").last().sort_index()
volume_crypto = cvol.resample("1D").sum().reindex(index=prices_crypto.index, columns=prices_crypto.columns)
ren = {c: c[:-4] if c.endswith("USDT") else c for c in prices_crypto.columns}
prices_crypto = prices_crypto.rename(columns=ren); volume_crypto = volume_crypto.rename(columns=ren)
prices_crypto.index.name = volume_crypto.index.name = "Date"

panel_crypto = DataPanel(prices_crypto, volume=volume_crypto, check_outliers=True)
print(f"Bars:   {len(panel_crypto.dates)}")
print(f"Assets: {len(panel_crypto.assets_all)}")
print(f"Range:  {panel_crypto.dates[0].date()} -> {panel_crypto.dates[-1].date()}")
print(f"NaN fraction: {prices_crypto.isna().mean().mean():.1%}")
live = prices_crypto.notna().sum(axis=1)
print(f"Coins live: {live.iloc[0]} at start -> {live.iloc[-1]} at end (breadth grows over time)")
dvol = (prices_crypto * volume_crypto).median()
q = dvol.quantile([0.05,0.5,0.95]) / 1e6
print(f"Median daily $-vol per coin ($M): p5={q.iloc[0]:.1f} median={q.iloc[1]:.1f} p95={q.iloc[2]:.0f}")


In [ ]:
# Cell 19 — Crypto cost stack
from backtest.costs import (
    Commission, Spread, MarketImpact, ShortBorrow,
    CompositeCostModel, LiquidityCap,
)
costs_crypto = CompositeCostModel([
    Commission(bps=5),                                  # ~5 bp/side (Binance VIP-ish)
    Spread(half_bps=5),                                 # blended majors+alts
    MarketImpact(k=12, kind="sqrt", adv_lookback=20),
    ShortBorrow(annual_bps=100, trading_days=365),      # nominal; perp funding is a wildcard
])
liq_crypto = LiquidityCap(cap_pct=0.05, adv_lookback=20)
print("Cost components:", [type(m).__name__ for m in costs_crypto.models])
print(f"panel_crypto has volume: {panel_crypto.has_field('volume')}")


In [ ]:
# Cell 20 — Crypto reversal: run + metrics (ann_factor=365)
strat_crypto = ReversalNeutral(lookback=5, clip=1.0, decay=5, neutralize=False)
result_crypto = Engine(panel_crypto, strat_crypto, costs=costs_crypto, liquidity=liq_crypto,
                       initial_capital=1_000_000).run(start=panel_crypto.dates[WARMUP_BARS])

rep_c = compute_metrics(result_crypto.returns, result_crypto.equity_curve,
                        costs=result_crypto.costs, trades=result_crypto.trades, ann_factor=365)
yrs_c = (result_crypto.equity_curve.index[-1] - result_crypto.equity_curve.index[0]).days / 365.25
gS_c = sharpe_ratio(result_crypto.gross_pnl / result_crypto.equity_curve.shift(1), ann_factor=365)

print(rep_c)
print()
print(f"NET   ret:    {result_crypto.equity_curve.iloc[-1]/1e6-1:+.2%}   over {yrs_c:.1f}y")
print(f"NET   Sharpe: {rep_c.sharpe:+.3f}  (95% CI [{rep_c.sharpe_ci_low:+.2f}, {rep_c.sharpe_ci_high:+.2f}])")
print(f"GROSS Sharpe: {gS_c:+.3f}   <- is reversal alive in crypto?")
print(f"Turnover:     {rep_c.turnover/yrs_c:.0f}x per year")
print(f"Margin:       {rep_c.margin*1e4:+.2f} bps net per $ traded")
print(f"PSR(0):       {rep_c.psr:.3f}   {'SIG' if rep_c.psr>0.95 else 'not sig'} at 95%")


In [ ]:
# Cell 21 — Crypto MOMENTUM = flipped reversal (trial #006)
class MomentumRanked(ReversalNeutral):
    """Long recent winners, short recent losers = negation of the reversal signal."""
    def __repr__(self):
        return (f"MomentumRanked(lookback={self.lookback}, clip={self.clip}, decay={self.decay})")
    def generate_weights(self, data, t):
        return -super().generate_weights(data, t)   # flip reversal -> momentum

strat_mom = MomentumRanked(lookback=5, clip=1.0, decay=5, neutralize=False)
result_mom = Engine(panel_crypto, strat_mom, costs=costs_crypto, liquidity=liq_crypto,
                    initial_capital=1_000_000).run(start=panel_crypto.dates[WARMUP_BARS])

rep_m = compute_metrics(result_mom.returns, result_mom.equity_curve,
                        costs=result_mom.costs, trades=result_mom.trades, ann_factor=365)
yrs_m = (result_mom.equity_curve.index[-1] - result_mom.equity_curve.index[0]).days / 365.25
gS_m = sharpe_ratio(result_mom.gross_pnl / result_mom.equity_curve.shift(1), ann_factor=365)

print(rep_m)
print()
print(f"NET   ret:    {result_mom.equity_curve.iloc[-1]/1e6-1:+.2%}   over {yrs_m:.1f}y")
print(f"NET   Sharpe: {rep_m.sharpe:+.3f}  (95% CI [{rep_m.sharpe_ci_low:+.2f}, {rep_m.sharpe_ci_high:+.2f}])")
print(f"GROSS Sharpe: {gS_m:+.3f}")
print(f"Turnover:     {rep_m.turnover/yrs_m:.0f}x/yr   Margin: {rep_m.margin*1e4:+.2f} bps/$")
print(f"PSR(0):       {rep_m.psr:.3f}   {'SIG at 95%' if rep_m.psr>0.95 else 'not sig'}")


In [ ]:
# Cell 22 — Crypto momentum: formation-horizon sweep (sensitivity, not cherry-pick)
SWEEP_WARMUP = 40                      # >= max lookback (30) + decay (5) for fair comparison
print("lookback | net Sh | gross Sh | net ret | turn/yr | PSR")
for lb in [3, 5, 10, 15, 20, 30]:
    s = MomentumRanked(lookback=lb, clip=1.0, decay=5, neutralize=False)
    r = Engine(panel_crypto, s, costs=costs_crypto, liquidity=liq_crypto,
               initial_capital=1_000_000).run(start=panel_crypto.dates[SWEEP_WARMUP])
    rep = compute_metrics(r.returns, r.equity_curve, costs=r.costs, trades=r.trades, ann_factor=365)
    yrs = (r.equity_curve.index[-1] - r.equity_curve.index[0]).days / 365.25
    gS = sharpe_ratio(r.gross_pnl / r.equity_curve.shift(1), ann_factor=365)
    print(f"  {lb:>4}d  | {rep.sharpe:>+5.2f}  | {gS:>+6.2f}   | "
          f"{r.equity_curve.iloc[-1]/1e6-1:>+6.1%} | {rep.turnover/yrs:>5.0f}x  | {rep.psr:.2f}")


In [ ]:
# Cell 23 — Extend horizon sweep to find the peak (crypto momentum)
EXT_WARMUP = 100                       # >= longest lookback (90) + decay; consistent start
print(f"start date: {panel_crypto.dates[EXT_WARMUP].date()}")
print("lookback | net Sh | gross Sh | net ret | turn/yr | PSR")
for lb in [30, 45, 60, 90]:
    s = MomentumRanked(lookback=lb, clip=1.0, decay=5, neutralize=False)
    r = Engine(panel_crypto, s, costs=costs_crypto, liquidity=liq_crypto,
               initial_capital=1_000_000).run(start=panel_crypto.dates[EXT_WARMUP])
    rep = compute_metrics(r.returns, r.equity_curve, costs=r.costs, trades=r.trades, ann_factor=365)
    yrs = (r.equity_curve.index[-1] - r.equity_curve.index[0]).days / 365.25
    gS = sharpe_ratio(r.gross_pnl / r.equity_curve.shift(1), ann_factor=365)
    print(f"  {lb:>4}d  | {rep.sharpe:>+5.2f}  | {gS:>+6.2f}   | "
          f"{r.equity_curve.iloc[-1]/1e6-1:>+7.1%} | {rep.turnover/yrs:>5.0f}x  | {rep.psr:.2f}")


In [ ]:
# Cell 24 — Per-year regime robustness for the chosen 30d crypto momentum
from backtest.metrics import yearly_metrics

strat_best = MomentumRanked(lookback=30, clip=1.0, decay=5, neutralize=False)
result_best = Engine(panel_crypto, strat_best, costs=costs_crypto, liquidity=liq_crypto,
                     initial_capital=1_000_000).run(start=panel_crypto.dates[40])

yr = yearly_metrics(result_best.returns, result_best.equity_curve, ann_factor=365)
# annualize per-year sharpe column is already annualized by ann_factor in helper
print("Per-calendar-year (net, after costs):")
print(yr.round(3).to_string())
print()
pos = (yr["sharpe"] > 0).sum()
print(f"Years with positive net Sharpe: {pos}/{len(yr)}")
print(f"Worst year: {yr['sharpe'].idxmin()} (Sharpe {yr['sharpe'].min():+.2f}, "
      f"return {yr.loc[yr['sharpe'].idxmin(),'total_return']:+.1%})")


In [ ]:
# Cell 25 — Fee sensitivity for 30d crypto momentum
scenarios = {
    "institutional": (2, 3),    # (commission_bps, spread_half_bps)
    "base/VIP":      (5, 5),
    "retail":        (10, 10),
}
print("scenario      | comm/spread | net Sh | net ret | margin bps/$")
for name, (cbp, sbp) in scenarios.items():
    cm = CompositeCostModel([
        Commission(bps=cbp), Spread(half_bps=sbp),
        MarketImpact(k=12, kind="sqrt", adv_lookback=20),
        ShortBorrow(annual_bps=100, trading_days=365),
    ])
    r = Engine(panel_crypto, MomentumRanked(lookback=30, clip=1.0, decay=5, neutralize=False),
               costs=cm, liquidity=liq_crypto, initial_capital=1_000_000).run(start=panel_crypto.dates[40])
    rep = compute_metrics(r.returns, r.equity_curve, costs=r.costs, trades=r.trades, ann_factor=365)
    print(f"{name:<13} | {cbp:>2}bp/{sbp:>2}bp   | {rep.sharpe:>+5.2f}  | "
          f"{r.equity_curve.iloc[-1]/1e6-1:>+7.1%} | {rep.margin*1e4:>+6.2f}")


In [ ]:
# Cell 26 — DSR deflation for the 30d crypto momentum candidate
import numpy as np
from backtest.selection import dsr, expected_max_sr
from backtest.metrics.core import sharpe_var_term
from backtest.metrics import psr as psr_fn

r = result_best.returns.dropna().to_numpy()
T = len(r)
sr_pb = r.mean() / r.std(ddof=1)
var_term = sharpe_var_term(r, sr_pb)
A = 365 ** 0.5

print(f"Realized Sharpe (ann.): {sr_pb*A:+.3f}   over {T} bars")
print(f"Undeflated PSR(0):      {psr_fn(result_best.returns):.3f}")
print(f"{'K':>4} | luck-max(ann) | DSR")
for K in [1, 9, 20]:
    lm = expected_max_sr(K, var_term / T) * A
    d = dsr(result_best.returns, K=K)
    print(f"{K:>4} | {lm:>+12.3f} | {d:.3f}  {'SIG' if d>0.95 else 'not sig'} at 95%")
print()
print("Note: K=9 treats all 9 horizon trials as independent (over-deflates, since they correlate).")
print("True effective-K is lower -> DSR between the K=1 and K=9 rows.")
